In [ ]:
import urllib.request
from pathlib import Path
import tarfile
import email
from email.utils import getaddresses
from collections import defaultdict
from pathlib import Path
import csv

In [ ]:
url = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
tarball = Path("enron_mail.tar.gz")

def show_progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    pct = min(downloaded / total_size * 100, 100)
    print(f"Downloading... {pct:.1f}% ({downloaded/1e6:.0f} MB / {total_size/1e6:.0f} MB)", end="")

if not tarball.exists():
    print("Downloading Enron email dataset...")
    urllib.request.urlretrieve(url, tarball, reporthook=show_progress)
    print(" Download complete.")
else:
    print(" Tarball already exists.")

In [ ]:
tarball = Path("enron_mail.tar.gz")
raw_csv = "enron_emails_raw.csv"
transformed_csv = "enron_emails_transformed.csv"
mapping_csv = "id_mapping.csv"

# ID Mapping
addr_to_id = {}
next_id = 1

def get_id(addr):
    global next_id
    addr = addr.lower().strip()
    if not addr or "@" not in addr: return None
    if addr not in addr_to_id:
        addr_to_id[addr] = next_id
        next_id += 1
    return addr_to_id[addr]

def get_id_list(header_text):
    """Parses a header string and returns a comma-separated string of IDs."""
    addresses = [addr for name, addr in getaddresses([header_text]) if addr]
    ids = [str(get_id(a)) for a in addresses if get_id(a)]
    return ",".join(ids)

print("Generating CSV files from Enron email dataset...")

# Create raw CSV with email headers
headers = ['From', 'To', 'Cc', 'Bcc', 'Subject', 'Date']

with tarfile.open(tarball, "r:gz") as tar:
    with open(raw_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        
        for i, member in enumerate(tar.getmembers()):
            if not member.isfile(): continue
            
            # Filter for the sent folder
            if not any(x in member.name.lower() for x in ["sent"]):
                continue

            try:
                efile = tar.extractfile(member)
                if efile:
                    msg = email.message_from_string(efile.read().decode('utf-8', errors='replace'))
                    writer.writerow({
                        'From': msg.get('From', ''),
                        'To': msg.get('To', ''),
                        'Cc': msg.get('Cc', ''),
                        'Bcc': msg.get('Bcc', ''),
                        'Subject': msg.get('Subject', ''),
                        'Date': msg.get('Date', '')
                    })
            except:
                continue
        print("Finished extracting emails to raw CSV.")
            
        

print(f"Created {raw_csv}")

# transform the raw CSV to replace email addresses with IDs
print("Transforming data to IDs...")

with open(raw_csv, 'r', encoding='utf-8') as f_in, \
     open(transformed_csv, 'w', newline='', encoding='utf-8') as f_out:
    
    reader = csv.DictReader(f_in)
    writer = csv.DictWriter(f_out, fieldnames=headers)
    writer.writeheader()
    
    for row in reader:
        writer.writerow({
            'From': get_id_list(row['From']),
            'To': get_id_list(row['To']),
            'Cc': get_id_list(row['Cc']),
            'Bcc': get_id_list(row['Bcc']),
            'Subject': row['Subject'],
            'Date': row['Date']
        })

# Write the ID mapping to a separate CSV
with open(mapping_csv, 'w', newline='', encoding='utf-8') as f_map:
    map_writer = csv.writer(f_map)
    map_writer.writerow(['Email_Address', 'ID'])
    for addr, uid in addr_to_id.items():
        map_writer.writerow([addr, uid])

print(f"Created {transformed_csv}")
print(f"Created {mapping_csv}")
